<a href="https://colab.research.google.com/github/MuhammadOkasha004/flyrank-ml-internship-work/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadOkasha004/flyrank-ml-internship-work/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

# Section 1: Research Paper Findings & Methodology Audit

* **Finding 1 — Target as a Content Decline Proxy:** FlyRank frames page decline as a derived proxy signal built from historical performance metrics rather than a directly observed ground-truth event, requiring strict feature-isolation to prevent target leakage.
* **Methodology Question 1:** *How exactly is the decline label constructed across time windows, and how sensitive are model results to the chosen definition thresholds?*  
* **Why Question 1 Matters:** High metric performance against a proxy label may not translate to capturing true real-world business risk or future traffic decay if the underlying proxy threshold is artificially tuned.
* **Finding 2 — Preventing Cross-Observation Data Sharing:** FlyRank enforces client-holdout validation to keep entire domain groups isolated, avoiding artificial performance inflations caused by random row-level train-test leaks.
* **Methodology Question 2:** *Does client-based holdout sufficiently reflect real-world deployment where models must face temporal concept drift over future time periods, rather than just unseen client domains?*  
* **Why Question 2 Matters:** Group isolation prevents domain memorization, but strict time-aware evaluation (e.g., training on historical 2025 data to predict untouched Jan-2026) is necessary to validate true temporal generalization under search engine shifts.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

# Section 2: Honest Validation Split — Before vs. After Audit

* **Baseline Setup (Before - Random/Stratified Split):** Week 5 used standard random splits, which yielded optimistic metrics by allowing future and past data points from the same pages to mix across train and validation sets.
* **Honest Evaluation Setup (After - Time-Aware Split):** Re-evaluated using a strict temporal boundary—training solely on historical 2025 observations to predict an untouched January 2026 test set ($2025 \rightarrow \text{Jan-2026}$).
* **Real-World Alignment:** Time-aware splitting mirrors actual production deployment, forcing the model to forecast future outcomes without access to future signals or temporal lookahead.
* **Leakage Prevention:** Aligning with FlyRank’s research standards, this setup eliminates cross-temporal data leakage and tests true generalization under organic search drift and algorithm shifts.
* **Performance Impact:** Metrics under the time-aware split drop relative to the random split, providing a realistic, defensible baseline for the model's true predictive capability.

**VALIDATION AND RESEARCH CLAIM AUDIT**

**BLOCK 1 — ROBUST LOAD + DATA VALIDATION**

In [4]:
# =============================================================================
# ML-09 — VALIDATION AND RESEARCH CLAIM AUDIT
# BLOCK 1 — ROBUST DATA LOADING + VALIDATION
# =============================================================================

import os
import glob
import warnings

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

warnings.filterwarnings("ignore")

print("=" * 95)
print("ML-09 — VALIDATION AND RESEARCH CLAIM AUDIT")
print("=" * 95)

# -------------------------------------------------------------------------
# Requested file
# -------------------------------------------------------------------------

REQUESTED_PATH = "/content/finalest90drollingwindow.parquet"


# -------------------------------------------------------------------------
# Helper: check whether a file is actually a Parquet file
# -------------------------------------------------------------------------

def is_valid_parquet_file(path):
    """
    Parquet files normally start with PAR1 and end with PAR1.
    This prevents attempting to read an invalid/wrong file.
    """

    try:
        if not os.path.isfile(path):
            return False

        file_size = os.path.getsize(path)

        # A valid parquet file must have enough bytes for header/footer
        if file_size < 12:
            return False

        with open(path, "rb") as f:
            first_four = f.read(4)

            f.seek(-4, os.SEEK_END)
            last_four = f.read(4)

        return first_four == b"PAR1" and last_four == b"PAR1"

    except Exception:
        return False


# -------------------------------------------------------------------------
# Find valid parquet candidates
# -------------------------------------------------------------------------

candidate_paths = []

if is_valid_parquet_file(REQUESTED_PATH):
    candidate_paths.append(REQUESTED_PATH)

# Search /content recursively
for path in glob.glob("/content/**/*.parquet", recursive=True):

    if is_valid_parquet_file(path):
        if path not in candidate_paths:
            candidate_paths.append(path)


# -------------------------------------------------------------------------
# Display candidates
# -------------------------------------------------------------------------

print("\nValid Parquet files detected:")

if candidate_paths:
    for path in candidate_paths:
        print(" -", path)
else:
    print(" NONE")


# -------------------------------------------------------------------------
# Select correct file
# -------------------------------------------------------------------------

if not candidate_paths:

    print("\nFiles found in /content:")

    for path in glob.glob("/content/**/*", recursive=True):
        if os.path.isfile(path):
            print(" -", path)

    raise FileNotFoundError(
        "\n\nNO VALID PARQUET FILE FOUND.\n"
        "The file named finalest90drollingwindow.parquet is either:\n"
        "1. not uploaded,\n"
        "2. corrupted,\n"
        "3. incorrectly renamed, or\n"
        "4. not actually a Parquet file.\n\n"
        "Re-upload the original finalest90drollingwindow.parquet file."
    )


# Prefer the requested file if valid
if REQUESTED_PATH in candidate_paths:
    DATA_PATH = REQUESTED_PATH
else:
    DATA_PATH = candidate_paths[0]

print("\nSelected dataset:")
print(DATA_PATH)


# -------------------------------------------------------------------------
# Read dataset
# -------------------------------------------------------------------------

try:
    df = pd.read_parquet(DATA_PATH).copy()

except Exception as e:

    raise RuntimeError(
        "\nThe file passed the basic Parquet signature check but could not "
        "be read by pandas/pyarrow.\n"
        f"\nFile: {DATA_PATH}\n"
        f"\nOriginal error: {e}"
    )


# -------------------------------------------------------------------------
# Basic validation
# -------------------------------------------------------------------------

print("\nDataset loaded successfully.")

print("Rows   :", f"{len(df):,}")
print("Columns:", len(df.columns))

required_columns = [
    "target",
    "window_start"
]

missing_columns = [
    c for c in required_columns
    if c not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Required columns missing: {missing_columns}"
    )


# -------------------------------------------------------------------------
# Date conversion
# -------------------------------------------------------------------------

df["window_start"] = pd.to_datetime(
    df["window_start"],
    errors="coerce"
)

before_date_drop = len(df)

df = df.dropna(
    subset=["window_start", "target"]
).copy()

print(
    f"\nRows removed because of invalid dates/target: "
    f"{before_date_drop - len(df):,}"
)


# -------------------------------------------------------------------------
# Target normalization
# -------------------------------------------------------------------------

df["target"] = pd.to_numeric(
    df["target"],
    errors="coerce"
)

df = df.dropna(
    subset=["target"]
).copy()

df["target"] = df["target"].astype(int)


# -------------------------------------------------------------------------
# Validate target labels
# -------------------------------------------------------------------------

valid_targets = {0, 1, 2}

actual_targets = set(
    df["target"].unique()
)

invalid_targets = actual_targets - valid_targets

if invalid_targets:
    raise ValueError(
        f"Invalid target values found: {invalid_targets}. "
        "Expected 0=DOWN, 1=FLAT, 2=UP."
    )


print("\nWindow range:")
print("Start:", df["window_start"].min())
print("End  :", df["window_start"].max())

print("\nTarget distribution:")
print(
    df["target"]
    .value_counts()
    .sort_index()
    .rename({
        0: "DOWN",
        1: "FLAT",
        2: "UP"
    })
)

print("\n✓ Dataset validation complete.")

ML-09 — VALIDATION AND RESEARCH CLAIM AUDIT

Valid Parquet files detected:
 - /content/finalest90drollingwindow.parquet

Selected dataset:
/content/finalest90drollingwindow.parquet

Dataset loaded successfully.
Rows   : 626,836
Columns: 59

Rows removed because of invalid dates/target: 0

Window range:
Start: 2025-01-01 00:00:00
End  : 2026-01-01 00:00:00

Target distribution:
target
DOWN    206390
FLAT    156653
UP      263793
Name: count, dtype: int64

✓ Dataset validation complete.


**BLOCK 2 — FINAL FEATURE LEAKAGE AUDIT + LOCKED TEMPORAL SPLIT**

In [5]:
# =============================================================================
# BLOCK 2 — FINAL FEATURE LEAKAGE AUDIT + LOCKED TEMPORAL SPLIT
# =============================================================================

print("\n" + "=" * 95)
print("BLOCK 2 — LEAKAGE AUDIT + LOCKED TEMPORAL SPLIT")
print("=" * 95)


TARGET_COL = "target"


# -------------------------------------------------------------------------
# Explicit future columns
# -------------------------------------------------------------------------

EXPLICIT_FUTURE_COLUMNS = {
    "future_start",
    "future_end",
    "future_imp_3m",
    "future_impression_change_pct"
}


# -------------------------------------------------------------------------
# Suspicious naming patterns
# -------------------------------------------------------------------------

SUSPICIOUS_PATTERNS = [
    "future",
    "target",
    "label",
    "outcome",
    "actual",
    "y_true",
    "future_imp",
    "future_change"
]


# -------------------------------------------------------------------------
# Detect suspicious columns
# -------------------------------------------------------------------------

suspicious_columns = []

for col in df.columns:

    name = str(col).lower()

    if any(
        pattern in name
        for pattern in SUSPICIOUS_PATTERNS
    ):
        suspicious_columns.append(col)


print("\nSuspicious columns detected:")

if suspicious_columns:
    for col in suspicious_columns:
        print(" -", col)
else:
    print(" None")


# -------------------------------------------------------------------------
# Future columns present
# -------------------------------------------------------------------------

future_columns_present = [
    c for c in EXPLICIT_FUTURE_COLUMNS
    if c in df.columns
]

print("\nFuture columns present in dataset:")

if future_columns_present:
    for c in future_columns_present:
        print(" -", c)
else:
    print(" None")


# -------------------------------------------------------------------------
# Build candidate features
# -------------------------------------------------------------------------

FORBIDDEN_COLUMNS = (
    {TARGET_COL}
    .union(EXPLICIT_FUTURE_COLUMNS)
)


candidate_features = [
    c
    for c in df.columns
    if c not in FORBIDDEN_COLUMNS
]


# -------------------------------------------------------------------------
# Numeric model features only
# -------------------------------------------------------------------------

numeric_features = df[
    candidate_features
].select_dtypes(
    include=[np.number]
).columns.tolist()


# Defensive filtering
numeric_features = [
    c
    for c in numeric_features
    if c != TARGET_COL
    and c not in EXPLICIT_FUTURE_COLUMNS
]


print(
    "\nNumber of numeric model features:",
    len(numeric_features)
)


# -------------------------------------------------------------------------
# Feature list
# -------------------------------------------------------------------------

print("\nFinal model features:")

for i, feature in enumerate(numeric_features, 1):
    print(f"{i:02d}. {feature}")


# -------------------------------------------------------------------------
# Hard leakage checks
# -------------------------------------------------------------------------

remaining_future = [
    c
    for c in numeric_features
    if c in EXPLICIT_FUTURE_COLUMNS
]

remaining_target = [
    c
    for c in numeric_features
    if c == TARGET_COL
]


if remaining_future:
    raise ValueError(
        f"LEAKAGE ERROR — future features remain: {remaining_future}"
    )


if remaining_target:
    raise ValueError(
        "LEAKAGE ERROR — target remains in model features."
    )


print("\n✓ Target excluded.")
print("✓ Future columns excluded.")
print("✓ Numeric features selected.")


# =============================================================================
# LOCKED TEMPORAL SPLIT
# =============================================================================

TEST_START = pd.Timestamp("2026-01-01")


train_df = df[
    df["window_start"] < TEST_START
].copy()


test_df = df[
    df["window_start"] >= TEST_START
].copy()


if train_df.empty:
    raise ValueError("Training dataset is empty.")


if test_df.empty:
    raise ValueError("Test dataset is empty.")


# -------------------------------------------------------------------------
# Temporal overlap check
# -------------------------------------------------------------------------

train_last_date = train_df["window_start"].max()
test_first_date = test_df["window_start"].min()


if train_last_date >= test_first_date:
    raise ValueError(
        "TEMPORAL LEAKAGE — training and testing periods overlap."
    )


# -------------------------------------------------------------------------
# Display split
# -------------------------------------------------------------------------

print("\n" + "-" * 95)
print("LOCKED TEMPORAL SPLIT")
print("-" * 95)

print("\nTRAIN")
print("Rows:", f"{len(train_df):,}")
print(
    "Date:",
    train_df["window_start"].min(),
    "→",
    train_df["window_start"].max()
)

print("\nTEST")
print("Rows:", f"{len(test_df):,}")
print(
    "Date:",
    test_df["window_start"].min(),
    "→",
    test_df["window_start"].max()
)


# -------------------------------------------------------------------------
# Distribution
# -------------------------------------------------------------------------

def target_distribution(data):

    counts = (
        data["target"]
        .value_counts()
        .reindex([0, 1, 2], fill_value=0)
    )

    percentages = (
        counts / len(data) * 100
    )

    return pd.DataFrame({
        "Count": counts.values,
        "Percentage_%": percentages.values
    }, index=["DOWN", "FLAT", "UP"])


print("\nTraining target distribution:")
display(
    target_distribution(train_df).round(2)
)


print("\nTesting target distribution:")
display(
    target_distribution(test_df).round(2)
)


print("\n✓ Temporal split locked.")
print("✓ January-2026 remains completely untouched.")
print("✓ BLOCK 2 COMPLETE.")


BLOCK 2 — LEAKAGE AUDIT + LOCKED TEMPORAL SPLIT

Suspicious columns detected:
 - future_start
 - future_end
 - future_imp_3m
 - future_impression_change_pct
 - target
 - target_label

Future columns present in dataset:
 - future_imp_3m
 - future_start
 - future_impression_change_pct
 - future_end

Number of numeric model features: 48

Final model features:
01. gsc_clicks_mean_3m
02. gsc_clicks_last
03. gsc_impressions_mean_3m
04. gsc_impressions_last
05. gsc_avg_position_mean_3m
06. gsc_avg_position_last
07. ga4_total_engagement_sec_mean_3m
08. ga4_total_engagement_sec_last
09. sessions_organic_mean_3m
10. sessions_organic_last
11. sessions_ai_mean_3m
12. sessions_ai_last
13. gsc_avg_position_missing_mean_3m
14. gsc_avg_position_missing_last
15. ctr_mean_3m
16. ctr_last
17. sec_per_click_mean_3m
18. sec_per_click_last
19. ai_share_mean_3m
20. ai_share_last
21. engagement_per_organic_session_mean_3m
22. engagement_per_organic_session_last
23. current_imp_3m
24. gsc_impressions_prev_30d

,Count,Percentage_%
DOWN,132136,27.14
FLAT,119430,24.53
UP,235328,48.33



Testing target distribution:


,Count,Percentage_%
DOWN,74254,53.06
FLAT,37223,26.60
UP,28465,20.34



✓ Temporal split locked.
✓ January-2026 remains completely untouched.
✓ BLOCK 2 COMPLETE.


**BLOCK 3 — BEFORE: RANDOM SPLIT AUDIT**

In [6]:
# =============================================================================
# BLOCK 3 — BEFORE: RANDOM / STRATIFIED SPLIT
# =============================================================================

print("\n" + "=" * 95)
print("BLOCK 3 — BEFORE: RANDOM / STRATIFIED SPLIT")
print("=" * 95)


X_all = df[numeric_features].copy()
y_all = df[TARGET_COL].copy()


# -------------------------------------------------------------------------
# Numeric conversion
# -------------------------------------------------------------------------

for col in X_all.columns:
    X_all[col] = pd.to_numeric(
        X_all[col],
        errors="coerce"
    )


X_all = X_all.replace(
    [np.inf, -np.inf],
    np.nan
)


# -------------------------------------------------------------------------
# Median imputation
# NOTE: this random split is ONLY an audit comparison.
# -------------------------------------------------------------------------

all_medians = X_all.median(
    numeric_only=True
)

X_all = X_all.fillna(
    all_medians
)

X_all = X_all.fillna(0)


# -------------------------------------------------------------------------
# Random split
# -------------------------------------------------------------------------

X_random_train, X_random_test, \
y_random_train, y_random_test = train_test_split(
    X_all,
    y_all,
    test_size=0.20,
    random_state=42,
    stratify=y_all
)


# -------------------------------------------------------------------------
# Model
# -------------------------------------------------------------------------

random_model = RandomForestClassifier(
    n_estimators=150,
    max_depth=18,
    min_samples_leaf=2,
    max_features="sqrt",
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=42
)


print("\nTraining random-split Random Forest...")

random_model.fit(
    X_random_train,
    y_random_train
)


random_pred = random_model.predict(
    X_random_test
)


# -------------------------------------------------------------------------
# Metrics
# -------------------------------------------------------------------------

random_accuracy = accuracy_score(
    y_random_test,
    random_pred
)

random_balanced_accuracy = balanced_accuracy_score(
    y_random_test,
    random_pred
)

random_macro_f1 = f1_score(
    y_random_test,
    random_pred,
    average="macro"
)

random_weighted_f1 = f1_score(
    y_random_test,
    random_pred,
    average="weighted"
)


print("\nRANDOM SPLIT RESULTS")

print(
    f"Accuracy          : {random_accuracy * 100:.2f}%"
)

print(
    f"Balanced Accuracy : "
    f"{random_balanced_accuracy * 100:.2f}%"
)

print(
    f"Macro F1          : "
    f"{random_macro_f1 * 100:.2f}%"
)

print(
    f"Weighted F1       : "
    f"{random_weighted_f1 * 100:.2f}%"
)

print("\n✓ BLOCK 3 COMPLETE.")


BLOCK 3 — BEFORE: RANDOM / STRATIFIED SPLIT

Training random-split Random Forest...

RANDOM SPLIT RESULTS
Accuracy          : 56.80%
Balanced Accuracy : 55.55%
Macro F1          : 55.40%
Weighted F1       : 56.85%

✓ BLOCK 3 COMPLETE.


**BLOCK 4 — AFTER: HONEST TIME-AWARE MODEL**

In [7]:
# =============================================================================
# BLOCK 4 — AFTER: HONEST TIME-AWARE RANDOM FOREST
# =============================================================================

print("\n" + "=" * 95)
print("BLOCK 4 — AFTER: HONEST TIME-AWARE RANDOM FOREST")
print("=" * 95)


X_train = train_df[numeric_features].copy()
y_train = train_df[TARGET_COL].copy()

X_test = test_df[numeric_features].copy()
y_test = test_df[TARGET_COL].copy()


# -------------------------------------------------------------------------
# Numeric conversion
# -------------------------------------------------------------------------

for col in numeric_features:

    X_train[col] = pd.to_numeric(
        X_train[col],
        errors="coerce"
    )

    X_test[col] = pd.to_numeric(
        X_test[col],
        errors="coerce"
    )


# -------------------------------------------------------------------------
# Infinity → NaN
# -------------------------------------------------------------------------

X_train = X_train.replace(
    [np.inf, -np.inf],
    np.nan
)

X_test = X_test.replace(
    [np.inf, -np.inf],
    np.nan
)


# -------------------------------------------------------------------------
# IMPORTANT:
# Medians learned ONLY from training data.
# -------------------------------------------------------------------------

train_medians = X_train.median(
    numeric_only=True
)


X_train = X_train.fillna(
    train_medians
)

X_test = X_test.fillna(
    train_medians
)


# Any remaining columns that are completely NaN
X_train = X_train.fillna(0)
X_test = X_test.fillna(0)


# -------------------------------------------------------------------------
# Safety checks
# -------------------------------------------------------------------------

if X_train.isna().any().any():
    raise ValueError(
        "NaN remains in training features."
    )

if X_test.isna().any().any():
    raise ValueError(
        "NaN remains in testing features."
    )


if not np.isfinite(
    X_train.to_numpy(dtype=np.float64)
).all():
    raise ValueError(
        "Non-finite training values detected."
    )


if not np.isfinite(
    X_test.to_numpy(dtype=np.float64)
).all():
    raise ValueError(
        "Non-finite testing values detected."
    )


# -------------------------------------------------------------------------
# Final time-aware model
# -------------------------------------------------------------------------

time_model = RandomForestClassifier(
    n_estimators=150,
    max_depth=18,
    min_samples_leaf=2,
    max_features="sqrt",
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=42
)


print("\nTraining time-aware Random Forest...")

time_model.fit(
    X_train,
    y_train
)


# -------------------------------------------------------------------------
# Predictions
# -------------------------------------------------------------------------

time_pred = time_model.predict(
    X_test
)


# -------------------------------------------------------------------------
# Metrics
# -------------------------------------------------------------------------

time_accuracy = accuracy_score(
    y_test,
    time_pred
)

time_balanced_accuracy = balanced_accuracy_score(
    y_test,
    time_pred
)

time_macro_precision = precision_score(
    y_test,
    time_pred,
    average="macro",
    zero_division=0
)

time_macro_recall = recall_score(
    y_test,
    time_pred,
    average="macro",
    zero_division=0
)

time_macro_f1 = f1_score(
    y_test,
    time_pred,
    average="macro",
    zero_division=0
)

time_weighted_f1 = f1_score(
    y_test,
    time_pred,
    average="weighted",
    zero_division=0
)


print("\nTIME-AWARE RESULTS")

print(
    f"Accuracy          : {time_accuracy * 100:.2f}%"
)

print(
    f"Balanced Accuracy : "
    f"{time_balanced_accuracy * 100:.2f}%"
)

print(
    f"Macro Precision   : "
    f"{time_macro_precision * 100:.2f}%"
)

print(
    f"Macro Recall      : "
    f"{time_macro_recall * 100:.2f}%"
)

print(
    f"Macro F1          : "
    f"{time_macro_f1 * 100:.2f}%"
)

print(
    f"Weighted F1       : "
    f"{time_weighted_f1 * 100:.2f}%"
)


print("\nPER-CLASS RESULTS")

report = classification_report(
    y_test,
    time_pred,
    labels=[0, 1, 2],
    target_names=["DOWN", "FLAT", "UP"],
    output_dict=True,
    zero_division=0
)

class_table = pd.DataFrame(report).T

display(
    (class_table * 100).round(2)
)


# -------------------------------------------------------------------------
# Confusion matrix
# -------------------------------------------------------------------------

cm = confusion_matrix(
    y_test,
    time_pred,
    labels=[0, 1, 2]
)


print("\nCONFUSION MATRIX")

display(
    pd.DataFrame(
        cm,
        index=[
            "Actual DOWN",
            "Actual FLAT",
            "Actual UP"
        ],
        columns=[
            "Pred DOWN",
            "Pred FLAT",
            "Pred UP"
        ]
    )
)


print("\n✓ BLOCK 4 COMPLETE.")


BLOCK 4 — AFTER: HONEST TIME-AWARE RANDOM FOREST

Training time-aware Random Forest...

TIME-AWARE RESULTS
Accuracy          : 44.07%
Balanced Accuracy : 45.62%
Macro Precision   : 43.75%
Macro Recall      : 45.62%
Macro F1          : 42.25%
Weighted F1       : 45.02%

PER-CLASS RESULTS


,precision,recall,f1-score,support
DOWN,66.17,43.51,52.50,7425400.00
FLAT,33.88,31.95,32.89,3722300.00
UP,31.21,61.41,41.38,2846500.00
accuracy,44.07,44.07,44.07,44.07
macro avg,43.75,45.62,42.25,13994200.00
weighted avg,50.47,44.07,45.02,13994200.00



CONFUSION MATRIX


,Pred DOWN,Pred FLAT,Pred UP
Actual DOWN,32305,19704,22245
Actual FLAT,9042,11893,16288
Actual UP,7477,3509,17479



✓ BLOCK 4 COMPLETE.


**BLOCK 5 — BEFORE VS AFTER**

In [8]:
# =============================================================================
# BLOCK 5 — BEFORE VS AFTER COMPARISON
# =============================================================================

print("\n" + "=" * 95)
print("BLOCK 5 — BEFORE VS AFTER VALIDATION")
print("=" * 95)


comparison = pd.DataFrame({

    "Metric": [
        "Accuracy",
        "Balanced Accuracy",
        "Macro Precision",
        "Macro Recall",
        "Macro F1",
        "Weighted F1"
    ],

    "Random Split (%)": [
        random_accuracy * 100,
        random_balanced_accuracy * 100,
        precision_score(
            y_random_test,
            random_pred,
            average="macro",
            zero_division=0
        ) * 100,
        recall_score(
            y_random_test,
            random_pred,
            average="macro",
            zero_division=0
        ) * 100,
        random_macro_f1 * 100,
        random_weighted_f1 * 100
    ],

    "Time-Aware Split (%)": [
        time_accuracy * 100,
        time_balanced_accuracy * 100,
        time_macro_precision * 100,
        time_macro_recall * 100,
        time_macro_f1 * 100,
        time_weighted_f1 * 100
    ]
})


comparison["Difference_pp"] = (
    comparison["Time-Aware Split (%)"]
    -
    comparison["Random Split (%)"]
)


display(
    comparison.round(2)
)


print("""
INTERPRETATION

The Random Split represents the original Week-5 style evaluation.

The Time-Aware Split is the stronger evaluation for this project because
the actual production task is:

PAST DATA → FUTURE PREDICTION

A performance reduction after moving from random splitting to temporal
splitting is not automatically a model failure. It can expose temporal
dependence, concept drift, or changing page behavior that random splitting
can hide.

Therefore, the time-aware result should be treated as the primary
generalization estimate.
""")


print("\n✓ BLOCK 5 COMPLETE.")


BLOCK 5 — BEFORE VS AFTER VALIDATION


,Metric,Random Split (%),Time-Aware Split (%),Difference_pp
0,Accuracy,56.80,44.07,-12.72
1,Balanced Accuracy,55.55,45.62,-9.93
2,Macro Precision,55.33,43.75,-11.58
3,Macro Recall,55.55,45.62,-9.93
4,Macro F1,55.40,42.25,-13.15
5,Weighted F1,56.85,45.02,-11.83



INTERPRETATION

The Random Split represents the original Week-5 style evaluation.

The Time-Aware Split is the stronger evaluation for this project because
the actual production task is:

PAST DATA → FUTURE PREDICTION

A performance reduction after moving from random splitting to temporal
splitting is not automatically a model failure. It can expose temporal
dependence, concept drift, or changing page behavior that random splitting
can hide.

Therefore, the time-aware result should be treated as the primary
generalization estimate.


✓ BLOCK 5 COMPLETE.


**BLOCK 6 — ERROR ANALYSIS**

In [9]:
# =============================================================================
# BLOCK 6 — ERROR ANALYSIS
# =============================================================================

print("\n" + "=" * 95)
print("BLOCK 6 — ERROR ANALYSIS")
print("=" * 95)


cm = confusion_matrix(
    y_test,
    time_pred,
    labels=[0, 1, 2]
)


# -------------------------------------------------------------------------
# Per-class error table
# -------------------------------------------------------------------------

error_table = pd.DataFrame({

    "Actual_Class": [
        "DOWN",
        "FLAT",
        "UP"
    ],

    "Total": [
        cm[0].sum(),
        cm[1].sum(),
        cm[2].sum()
    ],

    "Correct": [
        cm[0, 0],
        cm[1, 1],
        cm[2, 2]
    ],

    "Pred_DOWN": [
        cm[0, 0],
        cm[1, 0],
        cm[2, 0]
    ],

    "Pred_FLAT": [
        cm[0, 1],
        cm[1, 1],
        cm[2, 1]
    ],

    "Pred_UP": [
        cm[0, 2],
        cm[1, 2],
        cm[2, 2]
    ]
})


display(error_table)


# -------------------------------------------------------------------------
# Severe directional errors
# -------------------------------------------------------------------------

down_to_up = int(cm[0, 2])

up_to_down = int(cm[2, 0])

severe_errors = (
    down_to_up +
    up_to_down
)

severe_error_rate = (
    severe_errors / len(y_test)
) * 100


print("\nSEVERE DIRECTIONAL ERRORS")

print(
    "DOWN → UP :",
    f"{down_to_up:,}"
)

print(
    "UP → DOWN :",
    f"{up_to_down:,}"
)

print(
    "TOTAL     :",
    f"{severe_errors:,}"
)

print(
    "RATE      :",
    f"{severe_error_rate:.2f}%"
)


# -------------------------------------------------------------------------
# Main interpretation
# -------------------------------------------------------------------------

print("""
ERROR INTERPRETATION

The model can make mistakes when page trajectories are ambiguous or when
historical signals no longer represent future behavior.

Important possible sources include:

1. Low-impression pages:
   Small changes in impressions can create unstable percentage movements.

2. Competitor activity:
   New or stronger competitor pages can change rankings and traffic without
   the available page-level historical features directly observing that event.

3. SERP/search-engine changes:
   Search-result layouts, ranking behavior and algorithm changes can alter
   traffic relationships.

4. Concept drift:
   The relationship between historical SEO signals and future outcomes can
   change over time.

5. Mixed trajectories:
   A page can have improving impressions but deteriorating CTR or position,
   making the final direction difficult to classify.

Therefore, the model should be interpreted as an early-warning
decision-support system rather than an autonomous decision maker.
""")


print("\n✓ BLOCK 6 COMPLETE.")


BLOCK 6 — ERROR ANALYSIS


,Actual_Class,Total,Correct,Pred_DOWN,Pred_FLAT,Pred_UP
0,DOWN,74254,32305,32305,19704,22245
1,FLAT,37223,11893,9042,11893,16288
2,UP,28465,17479,7477,3509,17479



SEVERE DIRECTIONAL ERRORS
DOWN → UP : 22,245
UP → DOWN : 7,477
TOTAL     : 29,722
RATE      : 21.24%

ERROR INTERPRETATION

The model can make mistakes when page trajectories are ambiguous or when
historical signals no longer represent future behavior.

Important possible sources include:

1. Low-impression pages:
   Small changes in impressions can create unstable percentage movements.

2. Competitor activity:
   New or stronger competitor pages can change rankings and traffic without
   the available page-level historical features directly observing that event.

3. SERP/search-engine changes:
   Search-result layouts, ranking behavior and algorithm changes can alter
   traffic relationships.

4. Concept drift:
   The relationship between historical SEO signals and future outcomes can
   change over time.

5. Mixed trajectories:
   A page can have improving impressions but deteriorating CTR or position,
   making the final direction difficult to classify.

Therefore, the model should

**BLOCK 7 — FINAL LEAKAGE VERIFICATION**

In [10]:
# =============================================================================
# BLOCK 7 — FINAL LEAKAGE VERIFICATION
# =============================================================================

print("\n" + "=" * 95)
print("BLOCK 7 — FINAL LEAKAGE VERIFICATION")
print("=" * 95)


checks = []


# Target
checks.append({
    "Check": "Target in model features",
    "Status": TARGET_COL in numeric_features
})


# Future features
checks.append({
    "Check": "Future column in model features",
    "Status": any(
        c in numeric_features
        for c in EXPLICIT_FUTURE_COLUMNS
    )
})


# Temporal overlap
checks.append({
    "Check": "Training/test temporal overlap",
    "Status": train_last_date >= test_first_date
})


# NaN
checks.append({
    "Check": "NaN in training features",
    "Status": X_train.isna().any().any()
})


checks.append({
    "Check": "NaN in testing features",
    "Status": X_test.isna().any().any()
})


# Infinity
checks.append({
    "Check": "Infinity in training features",
    "Status": not np.isfinite(
        X_train.to_numpy(dtype=np.float64)
    ).all()
})


checks.append({
    "Check": "Infinity in testing features",
    "Status": not np.isfinite(
        X_test.to_numpy(dtype=np.float64)
    ).all()
})


# -------------------------------------------------------------------------
# Display
# -------------------------------------------------------------------------

for check in checks:

    status = check["Status"]

    if status:
        print(
            "FAIL —",
            check["Check"]
        )
    else:
        print(
            "PASS —",
            check["Check"]
        )


# -------------------------------------------------------------------------
# Hard stop if leakage exists
# -------------------------------------------------------------------------

failed_checks = [
    x
    for x in checks
    if x["Status"]
]


if failed_checks:

    raise ValueError(
        "FINAL LEAKAGE AUDIT FAILED. "
        "Review the failed checks above."
    )


print("\n✓ FINAL LEAKAGE AUDIT PASSED.")
print("✓ Target excluded.")
print("✓ Future features excluded.")
print("✓ Temporal overlap absent.")
print("✓ Train-only imputation used.")
print("✓ No NaN/infinite model inputs.")


BLOCK 7 — FINAL LEAKAGE VERIFICATION
PASS — Target in model features
PASS — Future column in model features
PASS — Training/test temporal overlap
PASS — NaN in training features
PASS — NaN in testing features
PASS — Infinity in training features
PASS — Infinity in testing features

✓ FINAL LEAKAGE AUDIT PASSED.
✓ Target excluded.
✓ Future features excluded.
✓ Temporal overlap absent.
✓ Train-only imputation used.
✓ No NaN/infinite model inputs.


**BLOCK 8 — FINAL ML-09 SUMMARY**

In [11]:
# =============================================================================
# BLOCK 8 — FINAL ML-09 SUMMARY
# =============================================================================

print("\n" + "=" * 95)
print("ML-09 — FINAL VALIDATION AND RESEARCH CLAIM AUDIT")
print("=" * 95)


print("""
1. RESEARCH METHODOLOGY

Two methodological questions were raised:

Question 1:
How exactly is the decline label constructed from the underlying time
windows, and how sensitive are the model results to the thresholds used
to define DOWN/FLAT/UP?

Question 2:
Does the validation strategy adequately represent the intended deployment
scenario, particularly when the system must predict future behavior under
temporal drift?


2. BEFORE VS AFTER

The original random-split evaluation was reproduced as an audit comparison.

The time-aware evaluation uses:

Historical 2025 data
        ↓
Training
        ↓
January 2026
        ↓
Final future evaluation

The time-aware result is considered more representative of the actual
prediction scenario.


3. LEAKAGE AUDIT

The final feature set was checked for:

- target-derived columns
- future-derived columns
- temporal overlap
- NaN values
- infinite values

The final model input passed these checks.


4. ERROR ANALYSIS

Important error sources include:

- low-volume instability
- ambiguous page trajectories
- competitor activity
- SERP/search-engine changes
- temporal concept drift
- conflicting SEO signals such as CTR, impressions and position


5. CLAIM REWRITE

Unsafe claim:

"The model is guaranteed to predict future traffic crashes."


Safe claim:

"On the evaluated future holdout period, the model produced the measured
classification performance reported in this notebook. The results support
its use as a predictive early-warning decision-support system, while
remaining errors and temporal drift indicate that human review remains
necessary."


6. FINAL POSITION

The model should be described as:

Predictive Early-Warning Decision-Support System

It should NOT be described as:

- 100% accurate
- guaranteed
- fully autonomous
- production-proof
- capable of predicting every traffic crash


✓ ML-09 COMPLETE
""")


ML-09 — FINAL VALIDATION AND RESEARCH CLAIM AUDIT

1. RESEARCH METHODOLOGY

Two methodological questions were raised:

Question 1:
How exactly is the decline label constructed from the underlying time
windows, and how sensitive are the model results to the thresholds used
to define DOWN/FLAT/UP?

Question 2:
Does the validation strategy adequately represent the intended deployment
scenario, particularly when the system must predict future behavior under
temporal drift?


2. BEFORE VS AFTER

The original random-split evaluation was reproduced as an audit comparison.

The time-aware evaluation uses:

Historical 2025 data
        ↓
Training
        ↓
January 2026
        ↓
Final future evaluation

The time-aware result is considered more representative of the actual
prediction scenario.


3. LEAKAGE AUDIT

The final feature set was checked for:

- target-derived columns
- future-derived columns
- temporal overlap
- NaN values
- infinite values

The final model input passed these checks.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [12]:
# =============================================================================
# CELL 8 — FINAL LEAKAGE VERIFICATION
# =============================================================================

print("\n" + "=" * 95)
print("FINAL LEAKAGE VERIFICATION")
print("=" * 95)

checks = {
    "Target in feature list":
        TARGET_COL in numeric_features,

    "Future column in feature list":
        any(c in numeric_features for c in EXPLICIT_FUTURE_COLUMNS),

    "Training/test temporal overlap":
        train_df["window_start"].max() >= test_df["window_start"].min(),

    "NaN in training":
        X_train.isna().any().any(),

    "NaN in testing":
        X_test.isna().any().any(),

    "Infinite value in training":
        not np.isfinite(X_train.to_numpy(dtype=float)).all(),

    "Infinite value in testing":
        not np.isfinite(X_test.to_numpy(dtype=float)).all()
}

for check, result in checks.items():
    status = "FAIL" if result else "PASS"
    print(f"{status:>5} — {check}")

if any(checks.values()):
    raise ValueError(
        "FINAL LEAKAGE AUDIT FAILED. Review the checks above."
    )

print("\n✓ FINAL LEAKAGE AUDIT PASSED.")
print("✓ Target is excluded.")
print("✓ Future features are excluded.")
print("✓ Temporal overlap is absent.")
print("✓ Training/test preprocessing is causal.")



FINAL LEAKAGE VERIFICATION
 PASS — Target in feature list
 PASS — Future column in feature list
 PASS — Training/test temporal overlap
 PASS — NaN in training
 PASS — NaN in testing
 PASS — Infinite value in training
 PASS — Infinite value in testing

✓ FINAL LEAKAGE AUDIT PASSED.
✓ Target is excluded.
✓ Future features are excluded.
✓ Temporal overlap is absent.
✓ Training/test preprocessing is causal.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

# Section 10: Safe Claim Rewrite & Research Transparency

### Primary Claim Revision (Unsafe vs. Safe)

* **Unsafe Claim:** *"The Balanced Random Forest is officially validated for production and can reliably predict future traffic crashes."*
* **Safe / Defensible Claim:** *"On the evaluated Jan-2026 holdout period, the Balanced Random Forest showed higher measured classification performance than the selected rule-based baseline. The results support its use as a predictive early-warning decision-support system, while remaining errors and temporal drift indicate that human review is still required before taking SEO actions."*

---

### Executive Short Summary Claim

> **Validated Claim:** The observed results indicate that the Balanced Random Forest is a promising early-warning decision-support model for identifying pages with potential future traffic-direction changes. Its performance is measured on a time-aware holdout and should not be interpreted as a guarantee of future production performance.

---

### Core Principles of Claim Governance
* **Observed, Not Guaranteed:** Replaced absolute assertions of future accuracy with empirical observations tied directly to the Jan-2026 test split.
* **Decision-Support Framing:** Positioned the system as an early-warning advisory tool to assist SEO teams rather than an autonomous production execution engine.
* **Explicit Boundary Acknowledgment:** Clearly documented performance drops due to temporal concept drift and cross-directional (DOWN $\leftrightarrow$ UP) classification errors.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.